# 04. Population and income (Version 3)

Who lives around each station? This notebook estimates the number of residents within walking distance of every station, adds household income by district, and uses **entries per resident** to tell home-end stations from destinations.

| Section | What it does |
|---|---|
| 0 | Settings |
| 1 | Download the population and income data (once, then cached) |
| 2 | Residents within the walking radius of each station |
| 3 | Entries per resident and station types |
| 4 | Checks: do the types make sense? |
| 5 | Household income by district |
| 6 | Save the outputs |

**The four station types**

| | Weekday-heavy | Weekend-leaning |
|---|---|---|
| **Serves its residents** (low entries per resident) | Home-end commuter | Local neighbourhood |
| **Draws visitors** (high entries per resident) | Work-end commuter | Leisure destination |

Needs the outputs of `01datapreparation.ipynb` and `03context1.ipynb` in `data/processed`.

## 0. Settings

In [29]:
from pathlib import Path
import gzip, json, shutil
from datetime import datetime
import requests
import pandas as pd
import geopandas as gpd

RADIUS_M = 500            # <- walking radius in metres (keep the same as 03context1 for consistency)
REFRESH_DOWNLOAD = False  # <- True = download the population and income files again

# Where to split the two axes of the station types. None = the network median (half the stations on each side).
RATIO_SPLIT = None               # <- weekend ratio below this = weekday-heavy, e.g. 0.6
ENTRIES_PER_RESIDENT_SPLIT = None # <- entries per resident above this = draws visitors, e.g. 1.0
MIN_RESIDENTS = 500              # <- stations with fewer residents than this are treated as destinations

TYPE_NAMES = {                   # <- names shown on the map for the four types
    ("weekday", "residents"): "Home-end commuter",
    ("weekday", "visitors"):  "Work-end commuter",
    ("weekend", "residents"): "Local neighbourhood",
    ("weekend", "visitors"):  "Leisure destination",
}

# Sources
KONTUR_URL = "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/kontur_population_MY_20231101.gpkg.gz"
INCOME_URL = "https://storage.dosm.gov.my/hies/hh_income_district.csv"

PROCESSED = Path("data/processed")
RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
KONTUR_FILE = RAW / "kontur_population_MY_20231101.gpkg"
INCOME_FILE = RAW / "hh_income_district.csv"
METRIC_CRS = "EPSG:32647"   # UTM zone 47N, measures in metres
print(f"Radius {RADIUS_M} m")

Radius 500 m


## 1. Download the data

**Population:** Kontur Population for Malaysia (1 November 2023), 400 m hexagons, CC BY 4.0. About 11 MB compressed; it is unpacked into `data/raw`.

**Income:** DOSM Household Income and Expenditure Survey, mean and median monthly household income by district.

In [30]:
def download(url, path):
    print(f"Downloading {url}")
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    path.write_bytes(r.content)

if REFRESH_DOWNLOAD or not KONTUR_FILE.exists():
    gz = KONTUR_FILE.with_suffix(".gpkg.gz")
    download(KONTUR_URL, gz)
    with gzip.open(gz, "rb") as src, open(KONTUR_FILE, "wb") as dst:
        shutil.copyfileobj(src, dst)
    gz.unlink()
else:
    print(f"Using cached {KONTUR_FILE}")

if REFRESH_DOWNLOAD or not INCOME_FILE.exists():
    download(INCOME_URL, INCOME_FILE)
else:
    print(f"Using cached {INCOME_FILE}")

Using cached data\raw\kontur_population_MY_20231101.gpkg
Using cached data\raw\hh_income_district.csv


## 2. Residents within walking distance

Each station gets a circle of `RADIUS_M`. Where a hexagon is only partly inside the circle, it contributes the same share of its population as the share of its area inside (this assumes people are spread evenly within a hexagon, which is the usual approach). Circles of nearby stations overlap, so a resident can count for two stations: that is correct, since they can walk to either.

In [31]:
stations = gpd.read_file(PROCESSED / "stations.geojson")
circles = stations[["key", "name", "geometry"]].to_crs(METRIC_CRS)
circles["geometry"] = circles.geometry.buffer(RADIUS_M)

# Read only the hexagons near the stations (the file covers all of Malaysia)
pop = gpd.read_file(KONTUR_FILE, bbox=tuple(circles.to_crs(gpd.read_file(KONTUR_FILE, rows=1).crs).total_bounds))
pop = pop.to_crs(METRIC_CRS)
pop["hex_area"] = pop.geometry.area
print(f"{len(pop):,} hexagons near the stations, {pop['population'].sum():,.0f} people in them")

pieces = gpd.overlay(circles[["key", "geometry"]], pop[["population", "hex_area", "geometry"]], how="intersection")
pieces["residents"] = pieces["population"] * pieces.geometry.area / pieces["hex_area"]
residents = pieces.groupby("key")["residents"].sum().round(-1)

data = stations[["key", "name", "district", "avg_daily_entries", "weekend_ratio"]].copy()
data["residents"] = data["key"].map(residents).fillna(0)
print(f"Median residents within {RADIUS_M} m: {data['residents'].median():,.0f}")
display(data.nlargest(8, "residents")[["name", "district", "residents", "avg_daily_entries"]])

1,641 hexagons near the stations, 6,913,734 people in them
Median residents within 500 m: 6,000


,name,district,residents,avg_daily_entries
4,Ampang,Ulu Langat,14550.0,3886.0
81,Pandan Indah,Ulu Langat,13010.0,2485.0
25,Cempaka,Ulu Langat,12710.0,3443.0
24,Cahaya,Ulu Langat,12690.0,2473.0
134,Taman Melati,W.P. Kuala Lumpur,11200.0,5563.0
152,Wangsa Maju,W.P. Kuala Lumpur,9820.0,8656.0
136,Taman Mutiara,W.P. Kuala Lumpur,8950.0,6591.0
105,Serdang Jaya,Petaling,8620.0,731.0


## 3. Entries per resident and station types

**Entries per resident** = average daily entries / residents within the radius. A value of 0.2 means one entry a day for every five residents: the station mostly serves its neighbourhood. A value of 5 means five entries for every resident: most users must come from elsewhere, to work, shop or visit.

In [32]:
data["entries_per_resident"] = (data["avg_daily_entries"] / data["residents"].where(data["residents"] >= MIN_RESIDENTS)).round(3)

ratio_split = RATIO_SPLIT if RATIO_SPLIT is not None else data["weekend_ratio"].median()
epr_split = ENTRIES_PER_RESIDENT_SPLIT if ENTRIES_PER_RESIDENT_SPLIT is not None else data["entries_per_resident"].median()
print(f"Splits used: weekend ratio {ratio_split:.2f}, entries per resident {epr_split:.2f}")

data["when"] = (data["weekend_ratio"] >= ratio_split).map({True: "weekend", False: "weekday"})
data["who"] = ((data["entries_per_resident"] > epr_split) | data["entries_per_resident"].isna()).map({True: "visitors", False: "residents"})
data["station_type"] = [TYPE_NAMES[(w, v)] for w, v in zip(data["when"], data["who"])]

print(data["station_type"].value_counts().to_string())
print(f"Stations with fewer than {MIN_RESIDENTS} residents (counted as destinations): {data['entries_per_resident'].isna().sum()}")

Splits used: weekend ratio 0.61, entries per resident 0.47
station_type
Work-end commuter      39
Leisure destination    39
Home-end commuter      38
Local neighbourhood    38
Stations with fewer than 500 residents (counted as destinations): 2


## 4. Checks

Stations whose character we already know should land in sensible types. The rank correlations then test the home-end idea from Version 2: if weekday-heavy stations include many residential ones, more residents should go with a lower weekend ratio.

In [33]:
known = ["kerinchi", "rajachulan", "klcc", "pasarseni", "bukitbintang", "imbi", "semantan", "sripetaling", "setiawangsa", "putraheights"]
show = ["name", "weekend_ratio", "residents", "avg_daily_entries", "entries_per_resident", "station_type"]
display(data.set_index("key").reindex([k for k in known if k in set(data["key"])])[show])

r = lambda a, b: data[a].rank().corr(data[b].rank())
print(f"Rank correlation, residents vs weekend ratio:            {r('residents', 'weekend_ratio'):+.2f}")
print(f"Rank correlation, residents vs entries:                  {r('residents', 'avg_daily_entries'):+.2f}")
print(f"Rank correlation, entries per resident vs weekend ratio: {r('entries_per_resident', 'weekend_ratio'):+.2f}")

for t in TYPE_NAMES.values():
    top = data[data["station_type"] == t].nlargest(5, "avg_daily_entries")["name"].tolist()
    print(f"\n{t}: {', '.join(top)}")

,name,weekend_ratio,residents,avg_daily_entries,entries_per_resident,station_type
key,,,,,,
kerinchi,Kerinchi,0.266,6220.0,6550.0,1.053,Work-end commuter
rajachulan,Raja Chulan,0.336,6330.0,3996.0,0.631,Work-end commuter
klcc,KLCC,0.663,6020.0,21710.0,3.606,Leisure destination
pasarseni,Pasar Seni,1.270,5470.0,19506.0,3.566,Leisure destination
bukitbintang,Bukit Bintang,1.006,7280.0,35031.0,4.812,Leisure destination
imbi,Imbi,1.168,7240.0,5096.0,0.704,Leisure destination
semantan,Semantan,0.323,6980.0,5228.0,0.749,Work-end commuter
sripetaling,Sri Petaling,0.538,4630.0,1215.0,0.262,Home-end commuter
setiawangsa,Setiawangsa,0.557,7390.0,3892.0,0.527,Work-end commuter


Rank correlation, residents vs weekend ratio:            +0.02
Rank correlation, residents vs entries:                  +0.14
Rank correlation, entries per resident vs weekend ratio: +0.01

Home-end commuter: Puchong Prima, Kinrara, Taman Paramount, Kelana Jaya, Bukit Dukung

Work-end commuter: Ampang Park, Abdullah Hukum, Muzium Negara, Wangsa Maju, Bank Rakyat Bangsar

Local neighbourhood: Ampang, Cempaka, Pandan Jaya, Kampung Baru, Kuchai

Leisure destination: Bukit Bintang, KL Sentral, KLCC, Pasar Seni, Bandar Utama


## 5. Household income by district

Each district's most recent year in the DOSM file (2024 for the Selangor districts; 2022 for Kuala Lumpur and Putrajaya, which DOSM has not yet published for 2024), joined to the district boundaries. Kuala Lumpur is one district, so this is broad context, not a station-level measure.

In [34]:
income = pd.read_csv(INCOME_FILE)
income["date"] = pd.to_datetime(income["date"])
# Use each district's most recent survey year: DOSM publishes the federal territories
# (Kuala Lumpur, Putrajaya) only for 2019 and 2022, so a single "latest year" would drop them
income = income.sort_values("date").groupby("district", as_index=False).last()
income["income_year"] = income["date"].dt.year
latest = income["date"].max()

def norm(s):
    s = str(s).lower().replace("w.p. ", "").replace("hulu ", "ulu ")
    return "".join(ch for ch in s if ch.isalnum())

districts = gpd.read_file(PROCESSED / "districts.geojson")
districts["match"] = districts["district"].map(norm)
income["match"] = income["district"].map(norm)
# Standardise the income column names (the file calls them e.g. income_mean / income_median)
rename = {c: ("income_median" if "median" in c.lower() else "income_mean")
          for c in income.columns if "mean" in c.lower() or "median" in c.lower()}
income = income.rename(columns=rename)
cols = [c for c in ["income_mean", "income_median"] if c in income.columns]
districts = districts.merge(income[["match"] + cols + ["income_year"]], on="match", how="left").drop(columns="match")
display(districts.drop(columns="geometry").sort_values(cols[-1], ascending=False))
missing = districts[districts[cols[-1]].isna()]["district"].tolist()
if missing:
    print("NO INCOME MATCH for:", missing, "- check the district names in the income file")

data = data.merge(districts[["district"] + cols + ["income_year"]], on="district", how="left")

,state,district,income_mean,income_median,income_year
4,Selangor,Ulu Langat,13822,11404,2024
3,Selangor,Sepang,15906,11131,2024
0,Selangor,Gombak,14168,10777,2024
2,Selangor,Petaling,14422,10688,2024
5,W.P. Kuala Lumpur,W.P. Kuala Lumpur,13325,10234,2022
6,W.P. Putrajaya,W.P. Putrajaya,13473,10056,2022
1,Selangor,Klang,10912,8837,2024


## 6. Save the outputs

In [35]:
out_cols = ["key", "residents", "entries_per_resident", "station_type"] + cols + ["income_year"]
data[out_cols].to_csv(PROCESSED / "station_population.csv", index=False)
districts.to_file(PROCESSED / "districts_income.geojson", driver="GeoJSON")

meta = {"radius_m": RADIUS_M, "ratio_split": round(float(ratio_split), 3),
        "entries_per_resident_split": round(float(epr_split), 3), "min_residents": MIN_RESIDENTS,
        "income_years": "most recent per district (2024 Selangor; 2022 Kuala Lumpur and Putrajaya)", "prepared_on": datetime.now().strftime("%Y-%m-%d"),
        "sources": {"population": "Kontur Population: Malaysia, 400 m H3 hexagons, 1 November 2023 (CC BY 4.0), via HDX",
                    "income": "DOSM, household income by administrative district (HIES)"},
        "type_names": [" / ".join(k) + " = " + v for k, v in TYPE_NAMES.items()]}
(PROCESSED / "population_metadata.json").write_text(json.dumps(meta, indent=2))

for f in ["station_population.csv", "districts_income.geojson", "population_metadata.json"]:
    print(f"{f:28} {(PROCESSED / f).stat().st_size / 1e3:8.0f} KB")

station_population.csv             10 KB
districts_income.geojson           30 KB
population_metadata.json            1 KB


___
**END**